# DermoGraph-XAI — DermoNet (Novel Architecture)
**Team 8 | VIT Bhopal | Skin Lesion Classification**

## Architecture Overview

| Component | Description |
|---|---|
| **DualScaleStem** | Parallel 3×3 (fine texture) + 7×7 (coarse shape) CNN branches with learned fusion weights |
| **LAAG ×2** | Lesion-Aware Attention Gate — channel attention (color) + spatial attention (borders) + border enhancement |
| **MRTB ×6** | Multi-Resolution Transformer Block — self-attention at 3 scales (fine+mid+coarse) simultaneously |
| Training | From scratch — no pretrained weights |

### Benchmark So Far
| Model | Acc | F1 | AUC |
|---|---|---|---|
| VGG16 | 80.48% | 0.7102 | 0.9601 |
| MobileNetV2 | 83.74% | 0.7334 | 0.9805 |
| ResNet50 | 87.40% | 0.7261 | 0.9823 |
| DenseNet121 | 87.69% | 0.7663 | 0.9866 |
| MaxViT-T | 91.98% | 0.8325 | 0.9840 |
| **DermoNet** | **?** | **?** | **?** |

## Cell 1 — Install & Verify GPU

In [ ]:
!pip install -q timm albumentations einops

import torch
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print("✓ GPU ready" if torch.cuda.is_available() else "✗ NO GPU")


## Cell 2 — Imports & Config

In [ ]:
import os, json, time, warnings, shutil, sys, math
warnings.filterwarnings('ignore')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (f1_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              precision_score, recall_score)

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT       = '/kaggle/working/'
CACHE_DIR    = '/kaggle/working/cache/'
BATCH_SIZE   = 32
N_CLASSES    = 7
N_EPOCHS     = 60
LR           = 3e-4
WEIGHT_DECAY = 1e-2
PATIENCE     = 20
GRAD_ACCUM   = 4
MODEL_NAME   = 'dermonet'
MEAN         = [0.485, 0.456, 0.406]
STD          = [0.229, 0.224, 0.225]
CLASS_NAMES  = ['Melanoma','Nevi','Basal Cell Carcinoma',
                'Actinic Keratosis','Benign Keratosis',
                'Dermatofibroma','Vascular']

torch.manual_seed(42)
np.random.seed(42)
os.makedirs(OUTPUT,    exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"✓ Config ready")
print(f"  Device     : {DEVICE}")
print(f"  Model      : DermoNet (novel — training from scratch)")
print(f"  Batch      : {BATCH_SIZE} × {GRAD_ACCUM} accum = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"  LR         : {LR}")
print(f"  Epochs     : {N_EPOCHS} (more patience — scratch training)")


## Cell 3 — Verify Paths

In [ ]:
print("=== ALL KAGGLE INPUT PATHS ===")
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth > 3: continue
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")


## Cell 4 — Load ALL 6 Datasets & Remap Paths

In [ ]:
BASE   = '/kaggle/input/datasets/akshat23029'
HAM1   = f'{BASE}/dermograph-ham-images/HAM10000_images_part_1'
HAM2   = f'{BASE}/dermograph-ham-images/HAM10000_images_part_2'
PAD1   = f'{BASE}/dermograph-pad-images/images/imgs_part_1'
PAD2   = f'{BASE}/dermograph-pad-images/images/imgs_part_2'
PAD3   = f'{BASE}/dermograph-pad-images/images/imgs_part_3'
SPLITS = f'{BASE}/dermograph-splits'
ISIC   = f'{BASE}/dermograph-isic2020'
MELAN  = f'{BASE}/dermograph-melanoma-cancer'
MIDAS  = f'{BASE}/dermograph-midas/midasmultimodalimagedatasetforaibasedskincancer'

print("Checking dataset paths...")
all_ok = True
for name, path in [
    ('HAM part1',       HAM1),
    ('HAM part2',       HAM2),
    ('PAD part1',       PAD1),
    ('PAD part2',       PAD2),
    ('PAD part3',       PAD3),
    ('Splits CSV',      SPLITS),
    ('ISIC 2020',       ISIC),
    ('Melanoma Cancer', MELAN),
    ('MIDAS',           MIDAS),
]:
    exists = os.path.exists(path)
    n      = len([f for f in os.listdir(path) if not f.startswith('.')]) if exists else 0
    print(f"  {'✓' if exists else '✗'} {name:<20}: {n:,} files")
    if not exists: all_ok = False

assert all_ok, "❌ Missing datasets! Add via + Add Input"

train_df = pd.read_csv(f'{SPLITS}/train.csv')
val_df   = pd.read_csv(f'{SPLITS}/val.csv')
test_df  = pd.read_csv(f'{SPLITS}/test.csv')
with open(f'{SPLITS}/class_weights.json') as f:
    cw_raw = json.load(f)

print(f"\nSplit sizes: Train={len(train_df):,}  Val={len(val_df):,}  Test={len(test_df):,}")

pad_lookup = {}
for part in [PAD1, PAD2, PAD3]:
    for fname in os.listdir(part):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            pad_lookup[fname] = f"{part}/{fname}"

isic_lookup = {}
for fname in os.listdir(ISIC):
    if fname.lower().endswith(('.jpg','.jpeg','.png')):
        isic_lookup[fname] = f"{ISIC}/{fname}"

midas_lookup = {}
for fname in os.listdir(MIDAS):
    if fname.lower().endswith(('.jpg','.jpeg','.png','.JPG','.JPEG')):
        midas_lookup[fname] = f"{MIDAS}/{fname}"

print(f"Lookups: PAD={len(pad_lookup):,}  ISIC={len(isic_lookup):,}  MIDAS={len(midas_lookup):,}")

def remap(mac_path):
    p = str(mac_path); fname = os.path.basename(p)
    if 'HAM10000_images_part_1' in p:
        fp = f"{HAM1}/{fname}"; return fp if os.path.exists(fp) else None
    if 'HAM10000_images_part_2' in p:
        fp = f"{HAM2}/{fname}"; return fp if os.path.exists(fp) else None
    if 'PAD-UFES' in p or 'pad' in p.lower() or 'imgs_part' in p:
        return pad_lookup.get(fname)
    if 'ISIC 2020' in p or 'isic' in p.lower():
        return isic_lookup.get(fname)
    if 'melanoma_cancer_dataset' in p:
        for split in ['train','test']:
            for cls in ['malignant','benign']:
                fp = f"{MELAN}/{split}/{cls}/{fname}"
                if os.path.exists(fp): return fp
        return None
    if 'MIDAS' in p or 'midas' in p.lower():
        return midas_lookup.get(fname)
    return None

print("\nRemapping paths...")
for name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df['kpath'] = df['image_path'].apply(remap)
    before = len(df)
    df.drop(df[df['kpath'].isna()].index, inplace=True)
    df.drop(df[df['label'] >= N_CLASSES].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    src = df['source'].value_counts().to_dict()
    print(f"  {name:<6}: {len(df):>6,} images (dropped {before-len(df)}) | {src}")
print("\n✓ All datasets loaded")


## Cell 5 — Cache Images

In [ ]:
all_paths = list(set(
    train_df['kpath'].tolist() + val_df['kpath'].tolist() + test_df['kpath'].tolist()
))
already = len([f for f in os.listdir(CACHE_DIR) if f.endswith('.jpg')])
print(f"Already cached: {already:,} | Total needed: {len(all_paths):,}\n")

done = skipped = errors = 0
for src_path in tqdm(all_paths, desc="Caching"):
    uid      = str(abs(hash(src_path)))[:10]
    dst_path = f"{CACHE_DIR}{uid}_{os.path.basename(src_path)}"
    if os.path.exists(dst_path): skipped += 1; continue
    img = cv2.imread(src_path)
    if img is None: errors += 1; continue
    img = cv2.resize(img, (224,224), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    done += 1

print(f"\n✓ Done: {done:,} new | {skipped:,} skipped | {errors} errors")
cache_lookup = {fname[:10]: f"{CACHE_DIR}{fname}" for fname in os.listdir(CACHE_DIR)}
def to_cache(p):
    uid = str(abs(hash(p)))[:10]
    return cache_lookup.get(uid, p)
for df in [train_df, val_df, test_df]:
    df['kpath'] = df['kpath'].apply(to_cache)
print("✓ Paths updated to cache")


## Cell 6 — Dataset & Augmentation

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=180, p=0.8),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=0.6),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.GaussNoise(p=0.3),
        A.CoarseDropout(max_holes=12, max_height=28, max_width=28, p=0.4),
        A.ElasticTransform(p=0.2),
        A.GridDistortion(p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class DermDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(str(row['kpath']))
        if img is None: img = np.zeros((224,224,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(image=img)['image']
        return img, torch.tensor(int(row['label']), dtype=torch.long)

img, lbl = DermDataset(train_df, get_train_transforms())[0]
print(f"✓ Dataset OK | img: {img.shape} | label: {lbl.item()} = {CLASS_NAMES[lbl.item()]}")


## Cell 7 — DataLoaders & Class Weights

In [ ]:
cw_list = [min(float(cw_raw.get(str(i), 1.0)), 10.0) for i in range(N_CLASSES)]
class_weights = torch.tensor(cw_list, dtype=torch.float32).to(DEVICE)

print("Class weights:")
for i, (name, w) in enumerate(zip(CLASS_NAMES, cw_list)):
    print(f"  {i} {name:<25} {w:.2f}  {'█'*int(w*2)}")

sampler = WeightedRandomSampler(
    weights     = torch.tensor(np.array(cw_list)[train_df['label'].values], dtype=torch.float32),
    num_samples = len(train_df),
    replacement = True,
)

train_loader = DataLoader(DermDataset(train_df, get_train_transforms()),
    batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(DermDataset(val_df,   get_val_transforms()),
    batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(DermDataset(test_df,  get_val_transforms()),
    batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✓ Loaders: Train={len(train_loader)} | Val={len(val_loader)} | Test={len(test_loader)}")


## Cell 8 — DermoNet Architecture

### Component 1: DualScaleStem — parallel fine(3×3) + coarse(7×7) branches
### Component 2: LAAG — Lesion-Aware Attention Gate (channel + spatial + border)
### Component 3: MRTB — Multi-Resolution Transformer Block (3 scales)

In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPONENT 1 — DualScaleStem
# Fine branch (3×3): hair, pores, fine texture
# Coarse branch (7×7): lesion boundary, overall shape
# Learned softmax fusion weights decide per-image balance
# ═══════════════════════════════════════════════════════════
class DualScaleStem(nn.Module):
    def __init__(self, out_channels=64):
        super().__init__()
        mid = out_channels // 2
        self.fine = nn.Sequential(
            nn.Conv2d(3, mid, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.GELU(),
            nn.Conv2d(mid, mid, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.GELU(),
        )
        self.coarse = nn.Sequential(
            nn.Conv2d(3, mid, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(mid), nn.GELU(),
            nn.Conv2d(mid, mid, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid), nn.GELU(),
        )
        self.fusion_weight = nn.Parameter(torch.ones(2) * 0.5)
        self.proj = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.GELU(),
        )

    def forward(self, x):
        w = torch.softmax(self.fusion_weight, dim=0)
        return self.proj(w[0]*self.fine(x) + w[1]*self.coarse(x))


# ═══════════════════════════════════════════════════════════
# COMPONENT 2 — LAAG (Lesion-Aware Attention Gate)
# Channel attention: WHICH features matter (color → melanoma)
# Spatial attention: WHERE to look (borders → BCC)
# Border enhancement: depthwise conv highlights lesion edges
# ═══════════════════════════════════════════════════════════
class LAAG(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels//reduction), nn.GELU(),
            nn.Linear(channels//reduction, channels), nn.Sigmoid(),
        )
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid(),
        )
        self.border_conv = nn.Conv2d(channels, channels, 3,
                                     padding=1, groups=channels, bias=False)

    def forward(self, x):
        x = x * self.channel_gate(x).unsqueeze(-1).unsqueeze(-1)
        avg = torch.mean(x, dim=1, keepdim=True)
        mx  = torch.max(x,  dim=1, keepdim=True)[0]
        x   = x * self.spatial_gate(torch.cat([avg, mx], dim=1))
        return x + 0.1 * self.border_conv(x)


# ═══════════════════════════════════════════════════════════
# COMPONENT 3 — MRTB (Multi-Resolution Transformer Block)
# Attention at 3 scales: fine(full) + mid(1/4) + coarse(1/16)
# Learned scale fusion + cross-scale enrichment
# ═══════════════════════════════════════════════════════════
class MultiHeadAttn(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.heads    = heads
        self.hd       = dim // heads
        self.scale    = self.hd ** -0.5
        self.qkv      = nn.Linear(dim, dim*3, bias=False)
        self.proj     = nn.Linear(dim, dim)
        self.drop     = nn.Dropout(dropout)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B,N,3,self.heads,self.hd).permute(2,0,3,1,4)
        q,k,v = qkv.unbind(0)
        attn  = self.drop((q@k.transpose(-2,-1)*self.scale).softmax(dim=-1))
        return self.proj((attn@v).transpose(1,2).reshape(B,N,C))

class MRTB(nn.Module):
    def __init__(self, dim, heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.attn_fine   = MultiHeadAttn(dim, heads, dropout)
        self.attn_mid    = MultiHeadAttn(dim, heads, dropout)
        self.attn_coarse = MultiHeadAttn(dim, heads, dropout)
        self.n1 = nn.LayerNorm(dim)
        self.n2 = nn.LayerNorm(dim)
        self.n3 = nn.LayerNorm(dim)
        self.scale_w    = nn.Parameter(torch.ones(3)/3)
        self.cross_proj = nn.Linear(dim, dim)
        mlp_d = int(dim*mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(dim, mlp_d), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(mlp_d, dim), nn.Dropout(dropout),
        )
        self.n_ffn = nn.LayerNorm(dim)

    def pool_tokens(self, x, target):
        B,N,C = x.shape
        if N==target: return x
        h=w=int(N**0.5); th=tw=int(target**0.5)
        return F.adaptive_avg_pool2d(
            x.transpose(1,2).reshape(B,C,h,w),(th,tw)
        ).reshape(B,C,-1).transpose(1,2)

    def upsample_tokens(self, x, h, w):
        B,N,C = x.shape
        sh=sw=int(N**0.5)
        return F.interpolate(
            x.transpose(1,2).reshape(B,C,sh,sw),
            size=(h,w), mode='bilinear', align_corners=False
        ).reshape(B,C,-1).transpose(1,2)

    def forward(self, x):
        B,N,C = x.shape
        h = w = int(N**0.5)
        fine   = x + self.attn_fine(self.n1(x))
        mid_in = self.pool_tokens(x, max(N//4,1))
        mid    = self.upsample_tokens(mid_in + self.attn_mid(self.n2(mid_in)), h, w)
        if N >= 16:
            crs_in = self.pool_tokens(x, max(N//16,1))
            coarse = self.upsample_tokens(crs_in + self.attn_coarse(self.n3(crs_in)), h, w)
        else:
            coarse = fine
        sw = torch.softmax(self.scale_w, dim=0)
        fused = sw[0]*fine + sw[1]*mid + sw[2]*coarse
        fused = fused + 0.1*self.cross_proj(fine-coarse)
        return fused + self.ffn(self.n_ffn(fused))


# ═══════════════════════════════════════════════════════════
# DERMONET — Full Model
# ═══════════════════════════════════════════════════════════
class DermoNet(nn.Module):
    def __init__(self, num_classes=7, embed_dim=256,
                 depth=6, heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        # Stage 1: Dual-Scale CNN Stem → (B,64,56,56)
        self.stem = DualScaleStem(out_channels=64)
        # Stage 2: CNN feature extractor with LAAG → (B,256,14,14)
        self.cnn_stage = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            LAAG(128),
            nn.Conv2d(128, embed_dim, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim), nn.GELU(),
            nn.Conv2d(embed_dim, embed_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim), nn.GELU(),
            LAAG(embed_dim),
        )
        # Stage 3: Tokenize
        self.patch_norm = nn.LayerNorm(embed_dim)
        self.pos_embed  = nn.Parameter(torch.randn(1,14*14,embed_dim)*0.02)
        self.pos_drop   = nn.Dropout(dropout)
        # Stage 4: MRTB stack
        self.blocks = nn.ModuleList([
            MRTB(embed_dim, heads, mlp_ratio, dropout) for _ in range(depth)
        ])
        # Stage 5: Head
        self.norm      = nn.LayerNorm(embed_dim)
        self.pre_logits= nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.Dropout(dropout)
        )
        self.head = nn.Linear(embed_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)                          # (B,64,56,56)
        x = self.cnn_stage(x)                     # (B,256,14,14)
        x = x.flatten(2).transpose(1,2)           # (B,196,256)
        x = self.pos_drop(self.patch_norm(x) + self.pos_embed)
        for blk in self.blocks: x = blk(x)
        x = self.norm(x).mean(dim=1)              # global avg pool
        return self.head(self.pre_logits(x))

print("✓ DermoNet architecture defined")
print("  1. DualScaleStem  — fine(3×3) + coarse(7×7) with learned fusion")
print("  2. LAAG ×2        — channel + spatial + border attention")
print("  3. MRTB ×6        — 3-scale transformer (fine+mid+coarse)")


## Cell 9 — Instantiate & Analyse DermoNet

In [ ]:
model = DermoNet(
    num_classes = N_CLASSES,
    embed_dim   = 256,
    depth       = 6,
    heads       = 8,
    mlp_ratio   = 4.0,
    dropout     = 0.1,
).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
print(f"✓ DermoNet ready")
print(f"  Total params : {total_p/1e6:.2f}M")
print(f"  MaxViT-T     : ~31M  |  DermoNet: {total_p/1e6:.2f}M")

with torch.no_grad():
    out = model(torch.randn(2,3,224,224).to(DEVICE))
print(f"  Output shape : {out.shape}  ✓")

print("\nParameter breakdown:")
for name, m in model.named_children():
    n = sum(p.numel() for p in m.parameters())
    print(f"  {name:<20}: {n/1e6:.3f}M")


## Cell 10 — Loss, Optimizer, Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight          = class_weights,
    label_smoothing = 0.1,
)

# Layer-wise LR: CNN parts learn faster than transformer
param_groups = [
    {'params': model.stem.parameters(),      'lr': LR*2.0},
    {'params': model.cnn_stage.parameters(), 'lr': LR*1.5},
    {'params': model.blocks.parameters(),    'lr': LR},
    {'params': model.head.parameters(),      'lr': LR*2.0},
]
optimizer = AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# Warm restarts help escape local minima when training from scratch
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)
scaler    = GradScaler()

print("✓ Loss / Optimizer / Scheduler ready")
print(f"  Loss      : CrossEntropyLoss + label_smoothing=0.1")
print(f"  Optimizer : AdamW with layer-wise LR")
print(f"    stem      : lr={LR*2:.1e}")
print(f"    cnn_stage : lr={LR*1.5:.1e}")
print(f"    blocks    : lr={LR:.1e}")
print(f"    head      : lr={LR*2:.1e}")
print(f"  Scheduler : CosineAnnealingWarmRestarts (T0=20, Tmult=2)")


## Cell 11 — Evaluate Function

In [ ]:
def evaluate(model, loader, desc='Evaluating'):
    model.eval()
    va_loss_sum = 0.0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc, leave=False, ncols=90):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs).float()          # FP32 — fixes AUC bug
            loss = criterion(out, labels)
            va_loss_sum += loss.item()
            probs = torch.softmax(out, dim=1).cpu().numpy().astype(np.float64)
            preds = out.argmax(dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
    all_probs  = np.vstack(all_probs).astype(np.float64)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs  = all_probs / all_probs.sum(axis=1, keepdims=True)  # row normalize
    va_loss = va_loss_sum / len(loader)
    va_acc  = float((all_preds == all_labels).mean())
    va_f1   = float(f1_score(all_labels, all_preds, average='macro', zero_division=0))
    try:
        va_auc = float(roc_auc_score(all_labels, all_probs,
                       multi_class='ovr', average='macro'))
    except Exception as e:
        print(f"  [AUC err]: {e}"); va_auc = 0.0
    return va_loss, va_acc, va_f1, va_auc, all_preds, all_labels, all_probs

print("✓ evaluate() ready — FP32 + row-normalized")


## Cell 12 — TRAIN DermoNet (60 Epochs)
> Training from SCRATCH — epoch 1-5 will show low accuracy (~0.3), climbs fast after epoch 8

In [ ]:
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk before training: {used/1e9:.1f}GB used / {free/1e9:.1f}GB free\n")

CKPT_PATH    = f"{OUTPUT}{MODEL_NAME}_best.pth"
best_val_acc = 0.0
best_val_auc = 0.0
no_improve   = 0
history      = []
start_total  = time.time()

print(f"Training DERMONET — {N_EPOCHS} epochs max, patience={PATIENCE}")
print(f"Training from SCRATCH — no pretrained weights")
print(f"Saving: {MODEL_NAME}_best.pth only")
print("=" * 85)
print(" Ep | TrainLoss |  ValLoss |  ValAcc |      F1 |     AUC |   Min | Status")
print("=" * 85)
sys.stdout.flush()

for epoch in range(N_EPOCHS):
    model.train()
    tr_loss_sum = 0.0
    optimizer.zero_grad()
    train_bar = tqdm(enumerate(train_loader), total=len(train_loader),
                     desc=f"Ep {epoch+1:>2}/{N_EPOCHS} [Train]", leave=False, ncols=90)
    for step, (imgs, labels) in train_bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            loss = criterion(model(imgs), labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % GRAD_ACCUM == 0 or (step+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        tr_loss_sum += loss.item() * GRAD_ACCUM
        train_bar.set_postfix(loss=f"{loss.item()*GRAD_ACCUM:.4f}")

    tr_loss = tr_loss_sum / len(train_loader)
    scheduler.step()

    va_loss, va_acc, va_f1, va_auc, _, _, _ = evaluate(
        model, val_loader, desc=f"Ep {epoch+1:>2}/{N_EPOCHS} [Val]  ")

    elapsed_min = (time.time() - start_total) / 60

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_val_auc = va_auc
        torch.save(model.state_dict(), CKPT_PATH)
        no_improve = 0
        status = "⭐ BEST"
    else:
        no_improve += 1
        status = f"pat {no_improve}/{PATIENCE}"

    print(f"{epoch+1:>3} | {tr_loss:>9.4f} | {va_loss:>8.4f} | "
          f"{va_acc:>7.4f} | {va_f1:>7.4f} | {va_auc:>7.4f} | "
          f"{elapsed_min:>5.1f} | {status}")
    sys.stdout.flush()

    history.append(dict(epoch=epoch+1, tr_loss=tr_loss,
                        va_loss=va_loss, va_acc=va_acc,
                        va_f1=va_f1, va_auc=va_auc))

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

total_min = (time.time() - start_total) / 60
print("=" * 85)
print(f"Training done in {total_min:.1f} min")
print(f"  Best Val Acc : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC : {best_val_auc:.4f}")
print(f"  Epochs run   : {len(history)}")
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"  Disk used    : {used/1e9:.1f}GB / {free/1e9:.1f}GB free")

with open(f"{OUTPUT}{MODEL_NAME}_history.json", 'w') as f:
    json.dump(history, f, indent=2)
print("✓ History saved")


## Cell 13 — Training Curves

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle('DermoNet — Training Curves (Novel Architecture)', fontsize=14, fontweight='bold')

axes[0].plot(hist_df['epoch'], hist_df['tr_loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(hist_df['epoch'], hist_df['va_loss'], label='Val Loss',   color='#2ecc71')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_df['epoch'], hist_df['va_acc'], label='Val Acc', color='#2ecc71')
axes[1].axhline(y=best_val_acc, color='gold', linestyle='--', label=f'Best {best_val_acc:.4f}')
axes[1].axhline(y=0.9198, color='red', linestyle=':', alpha=0.5, label='MaxViT-T 91.98%')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(hist_df['epoch'], hist_df['va_auc'], label='Val AUC', color='#3498db')
axes[2].axhline(y=best_val_auc, color='gold', linestyle='--', label=f'Best {best_val_auc:.4f}')
axes[2].axhline(y=0.9840, color='red', linestyle=':', alpha=0.5, label='MaxViT-T 0.9840')
axes[2].set_title('AUC-ROC'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {MODEL_NAME}_curves.png")


## Cell 14 — Final Test Evaluation

In [ ]:
print("Loading best checkpoint...")
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

ts_loss, ts_acc, ts_f1, ts_auc, ts_preds, ts_labels, ts_probs = evaluate(
    model, test_loader, desc="Testing"
)

print("\n" + "=" * 62)
print(f"  DERMONET — FINAL TEST RESULTS")
print("=" * 62)
print(f"  Test Accuracy : {ts_acc*100:.2f}%")
print(f"  Test F1 Macro : {ts_f1:.4f}")
print(f"  Test AUC-ROC  : {ts_auc:.4f}")
print(f"  Best Val Acc  : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC  : {best_val_auc:.4f}")
print(f"  Epochs run    : {len(history)}")
print(f"  Params        : {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print("=" * 62)

print("\nPer-class Report:")
print(classification_report(ts_labels, ts_preds, target_names=CLASS_NAMES, digits=2))

print("\n" + "=" * 62)
print("  FULL BENCHMARK")
print("=" * 62)
print(f"  {'Model':<22} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print(f"  {'-'*22} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'VGG16':<22} {'80.48%':>8} {'0.7102':>8} {'0.9601':>8}")
print(f"  {'MobileNetV2':<22} {'83.74%':>8} {'0.7334':>8} {'0.9805':>8}")
print(f"  {'ResNet50':<22} {'87.40%':>8} {'0.7261':>8} {'0.9823':>8}")
print(f"  {'DenseNet121':<22} {'87.69%':>8} {'0.7663':>8} {'0.9866':>8}")
print(f"  {'MaxViT-T':<22} {'91.98%':>8} {'0.8325':>8} {'0.9840':>8}")
print(f"  {'DermoNet (ours) ★':<22} {ts_acc*100:>7.2f}% {ts_f1:>8.4f} {ts_auc:>8.4f}")
print("=" * 62)


## Cell 15 — Confusion Matrix (Viridis)

In [ ]:
cm    = confusion_matrix(ts_labels, ts_preds)
SHORT = ['MEL','NV','BCC','AK','BKL','DF','VASC']

for title, data, fname, norm in [
    ('DermoNet (Novel)', cm, 'cm_viridis', False),
    ('DermoNet (Novel) — Normalized',
     cm.astype(float)/cm.sum(axis=1,keepdims=True), 'cm_normalized', True),
]:
    fig, ax = plt.subplots(figsize=(9,7))
    im   = ax.imshow(data, cmap='viridis', aspect='auto',
                     vmin=0, vmax=(1 if norm else None))
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=11)
    for i in range(len(SHORT)):
        for j in range(len(SHORT)):
            v     = data[i,j]
            thresh= 0.6 if norm else data.max()*0.6
            color = 'white' if v < thresh else 'black'
            ax.text(j, i, f'{v:.2f}' if norm else str(int(v)),
                    ha='center', va='center',
                    fontsize=13 if norm else 14, fontweight='bold', color=color)
    ax.set_xticks(np.arange(len(SHORT))); ax.set_yticks(np.arange(len(SHORT)))
    ax.set_xticklabels(SHORT, fontsize=12, fontweight='bold')
    ax.set_yticklabels(SHORT, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted label', fontsize=13, fontweight='bold', labelpad=10)
    ax.set_ylabel('True label',      fontsize=13, fontweight='bold', labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(np.arange(len(SHORT))-0.5, minor=True)
    ax.set_yticks(np.arange(len(SHORT))-0.5, minor=True)
    ax.grid(which='minor', color='white', linewidth=2)
    ax.tick_params(which='minor', length=0)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT}{MODEL_NAME}_{fname}.png", dpi=150, bbox_inches='tight')
    plt.show()
print("✓ Confusion matrices saved")


## Cell 16 — TP/FP/FN/TN per Class

In [ ]:
COL_TP='#2ecc71'; COL_FN='#e74c3c'; COL_FP='#e6a817'; COL_TN='#1abc9c'; BG='#1a1a2e'

total_n = len(ts_labels)
per = []
for i in range(N_CLASSES):
    TP=int(cm[i,i]); FN=int(cm[i,:].sum()-TP)
    FP=int(cm[:,i].sum()-TP); TN=int(total_n-TP-FN-FP)
    per.append((TP,FN,FP,TN,int(cm[i,:].sum())))

rows_idx=[0,0,0,0,1,1,1]; cols_idx=[0,1,2,3,0,1,2]; n_cols=4
fig = plt.figure(figsize=(22,14), facecolor=BG)
fig.patch.set_facecolor(BG)
fig.text(0.5,0.97,'DERMONET (Novel) — TP/FP/FN/TN per Class',
         ha='center',va='top',fontsize=18,fontweight='bold',color='white')
fig.text(0.5,0.93,f'Acc={ts_acc*100:.2f}%   F1={ts_f1:.4f}   AUC={ts_auc:.4f}',
         ha='center',va='top',fontsize=14,fontweight='bold',color='white')

lm=0.04; rm=0.78; tm=0.88; rh=0.38
cw_=(rm-lm)/n_cols; cew=cw_*0.42; ceh=rh*0.42

def draw_block(fig,cx,cy,tp,fn,fp,tn,cname,sup):
    fig.text(cx+cw_/2,cy+rh*0.96,cname,ha='center',va='top',
             fontsize=11,fontweight='bold',color='white')
    fig.text(cx+cw_/2,cy+rh*0.89,f'(n={sup})',ha='center',va='top',
             fontsize=9,color='#aaaaaa')
    px=cw_*0.04; py=rh*0.06; gap=cw_*0.02
    for qc,qr,val,color,label in [
        (0,1,tp,COL_TP,'TP'),(1,1,fn,COL_FN,'FN'),
        (0,0,fp,COL_FP,'FP'),(1,0,tn,COL_TN,'TN')]:
        x=cx+px+qc*(cew+gap); y=cy+py+qr*(ceh+gap)
        ax=fig.add_axes([x,y,cew,ceh])
        ax.set_facecolor(color); ax.set_xlim(0,1); ax.set_ylim(0,1)
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        ax.text(0.5,0.58,str(val),ha='center',va='center',
                fontsize=20,fontweight='bold',color='white',transform=ax.transAxes)
        ax.text(0.5,0.18,label,ha='center',va='center',
                fontsize=11,fontweight='bold',color='white',transform=ax.transAxes)
    fig.text(cx+px-0.005,cy+py+1.5*ceh+gap,'Act\nNeg',
             ha='right',va='center',fontsize=7.5,color='#cccccc')
    fig.text(cx+px-0.005,cy+py+0.5*ceh,'Act\nPos',
             ha='right',va='center',fontsize=7.5,color='#cccccc')
    fig.text(cx+px+0.5*cew,cy+py-0.025,'Pred\nPos',
             ha='center',va='top',fontsize=7.5,color='#cccccc')
    fig.text(cx+px+1.5*cew+gap,cy+py-0.025,'Pred\nNeg',
             ha='center',va='top',fontsize=7.5,color='#cccccc')

for i,(cname,(tp,fn,fp,tn,sup)) in enumerate(zip(CLASS_NAMES,per)):
    draw_block(fig, lm+cols_idx[i]*cw_, tm-(rows_idx[i]+1)*rh+0.02,
               tp,fn,fp,tn,cname,sup)

leg_x=0.80; leg_y=0.55
fig.text(leg_x+0.04,leg_y+0.06,'Legend',ha='center',va='bottom',
         fontsize=13,fontweight='bold',color='white')
for k,(color,label,desc) in enumerate([
    (COL_TP,'TP','Correctly detected'),(COL_FN,'FN','Missed / not detected'),
    (COL_FP,'FP','Wrongly detected'),(COL_TN,'TN','Correctly rejected')]):
    y=leg_y-k*0.07
    ax_leg=fig.add_axes([leg_x,y-0.025,0.035,0.045])
    ax_leg.set_facecolor(color); ax_leg.set_xticks([]); ax_leg.set_yticks([])
    for sp in ax_leg.spines.values(): sp.set_visible(False)
    ax_leg.text(0.5,0.5,label,ha='center',va='center',
                fontsize=10,fontweight='bold',color='white',transform=ax_leg.transAxes)
    fig.text(leg_x+0.045,y,desc,ha='left',va='center',fontsize=10,color='#cccccc')

plt.savefig(f"{OUTPUT}{MODEL_NAME}_tp_fp_fn_tn.png",dpi=150,bbox_inches='tight',facecolor=BG)
plt.show()
print(f"✓ Saved: {MODEL_NAME}_tp_fp_fn_tn.png")


## Cell 17 — Save Results JSON & Final Summary

In [ ]:
per_precision = precision_score(ts_labels, ts_preds, average=None, zero_division=0)
per_recall    = recall_score(ts_labels, ts_preds, average=None, zero_division=0)
per_f1_vals   = f1_score(ts_labels, ts_preds, average=None, zero_division=0)

results = {
    'model'       : 'DermoNet',
    'type'        : 'novel_architecture',
    'description' : 'DualScaleStem + LAAG×2 + MRTB×6 — trained from scratch',
    'params_M'    : round(sum(p.numel() for p in model.parameters())/1e6, 2),
    'test_accuracy': float(ts_acc),
    'test_f1_macro': float(ts_f1),
    'test_auc_roc' : float(ts_auc),
    'best_val_acc' : float(best_val_acc),
    'best_val_auc' : float(best_val_auc),
    'epochs_run'   : len(history),
    'components'   : [
        'DualScaleStem: parallel 3x3+7x7 CNN with learned fusion',
        'LAAG x2: channel+spatial+border attention gate',
        'MRTB x6: multi-resolution transformer (3 scales)',
    ],
    'per_class': {
        name: {'precision':float(per_precision[i]),
               'recall':float(per_recall[i]),
               'f1':float(per_f1_vals[i])}
        for i,name in enumerate(CLASS_NAMES)
    },
    'history': history,
}

with open(f"{OUTPUT}{MODEL_NAME}_results.json",'w') as f:
    json.dump(results, f, indent=2)

torch.cuda.empty_cache()
total, used, free = shutil.disk_usage("/kaggle/working")

print("=" * 62)
print(f"  ✓ DERMONET COMPLETE")
print("=" * 62)
print(f"  Test Acc   : {ts_acc*100:.2f}%")
print(f"  Test F1    : {ts_f1:.4f}")
print(f"  Test AUC   : {ts_auc:.4f}")
print(f"  Params     : {results['params_M']}M")
print(f"  Disk used  : {used/1e9:.1f}GB / {free/1e9:.1f}GB free")
print("=" * 62)
print("\n  ⚠️  SAVE VERSION NOW before closing!")
print("\n  Files to download:")
for fname in sorted(os.listdir(OUTPUT)):
    if MODEL_NAME in fname and not fname.endswith('.ipynb'):
        sz = os.path.getsize(f'{OUTPUT}/{fname}')/1024
        print(f"    {fname:<45} {'%.0f KB'%sz if sz<1024 else '%.1f MB'%(sz/1024)}")
print("\n  BENCHMARK:")
print(f"  VGG16          80.48%  F1=0.7102  AUC=0.9601")
print(f"  MobileNetV2    83.74%  F1=0.7334  AUC=0.9805")
print(f"  ResNet50       87.40%  F1=0.7261  AUC=0.9823")
print(f"  DenseNet121    87.69%  F1=0.7663  AUC=0.9866")
print(f"  MaxViT-T       91.98%  F1=0.8325  AUC=0.9840")
print(f"  DermoNet(ours) {ts_acc*100:.2f}%  F1={ts_f1:.4f}  AUC={ts_auc:.4f}")
